# K-Means Time Series Analysis on Korean Romance Films

Here I want to analyze the variation of color throughout the length of films. Basically as before splitting a film into 5 segments, applying k_means to that segments, getting 15 ish colors, but unlike before i won't group all the colors together, rather ill analyze each segment separately.

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML 
import matplotlib.colors as mcolors
import math
from collections import Counter
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

### Load the data and preprocess

In [2]:
# Path to the directory containing the CSV files
directory_path = '/Users/rsudhir/Documents/GitHub/Data-Science-Project---Colors-Of-Romance/Korean-Analysis/Korean-Movie-CSVs'

# List to hold each DataFrame
dfs = []

# Loop through the files in the directory and load each CSV
for i in range(1, 26):
    file_path = os.path.join(directory_path, f'{i}.csv')
    df = pd.read_csv(file_path)
    # Drop columns with NaN values
    df = df.drop(columns=['color_10_r', 'color_10_g', 'color_10_b'])
    
    # Add a column to indicate which movie the data is from
    df['movie_id'] = i
    
    dfs.append(df)

# Combine all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Display the combined DataFrame to ensure it looks correct
display(combined_df.head())

,frame_path,color_1_r,color_1_g,color_1_b,color_2_r,color_2_g,color_2_b,color_3_r,color_3_g,color_3_b,...,color_7_r,color_7_g,color_7_b,color_8_r,color_8_g,color_8_b,color_9_r,color_9_g,color_9_b,movie_id
0,output_0000001.png,53,53,38,169,195,201,112,123,116,...,130,139,148,76,100,94,116,132,84,1
1,output_0000002.png,106,119,107,150,153,145,38,41,34,...,57,66,41,44,60,55,156,172,156,1
2,output_0000003.png,187,210,188,77,76,71,31,35,36,...,120,139,132,108,92,75,60,84,76,1
3,output_0000004.png,214,228,250,245,245,249,184,202,250,...,186,212,251,188,183,236,215,211,211,1
4,output_0000005.png,117,111,98,17,18,16,172,165,166,...,145,139,148,83,82,85,156,108,94,1


In [3]:
# Define the number of segments
num_segments = 5

# Loop through each movie and create a segment ID
combined_df['segment_id'] = combined_df.groupby('movie_id').cumcount() // (combined_df.groupby('movie_id')['frame_path'].transform('count') // num_segments)

### K_means on each segment

For each segment of each movie i am getting x colors

In [4]:
# Apply K-Means Clustering to Each Segment Separately
num_clusters_per_segment = 15  # Number of clusters per segment

# Create a list to hold cluster centers for each segment
segment_clusters = []

# Group by movie and segment
grouped = combined_df.groupby(['movie_id', 'segment_id'])

for (movie_id, segment_id), group in grouped:
    # Extract RGB values for the dominant colors in this segment
    colors = []
    for i in range(1, 10):
        colors.append(group[[f'color_{i}_r', f'color_{i}_g', f'color_{i}_b']].dropna().values)
    
    # Combine colors into a single array
    colors_array = np.vstack(colors)

    # Ensure sufficient data points for clustering
    if len(colors_array) >= num_clusters_per_segment:
        # Perform K-Means clustering
        kmeans = KMeans(n_clusters=num_clusters_per_segment, n_init='auto', random_state=42)
        kmeans.fit(colors_array)
        
        # Store the cluster centers for this segment
        segment_clusters.append((movie_id, segment_id, kmeans.cluster_centers_))
    else:
        print(f"Skipping segment {segment_id} of movie {movie_id} due to insufficient data points.")


Skipping segment 5 of movie 1 due to insufficient data points.


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1152: ConvergenceWarning: Number of distinct clusters (11) found smaller than n_clusters (15). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1152: ConvergenceWarning: Number of distinct clusters (11) found smaller than n_clusters (15). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/base.py:1152: ConvergenceWarning: Number of distinct clusters (11) found smaller than n_clusters (15). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Skipping segment 5 of movie 21 due to insufficient data points.
Skipping segment 5 of movie 22 due to insufficient data points.


### Visualization of all segments and their colors

Have commented out the code as its quite a long output, feel free to uncomment if you'd like to see all the colors.

In [5]:
# # Visualization and Analysis
# for movie_id, segment_id, centers in segment_clusters:
#     # Convert RGB values to HEX
#     hex_colors = [mcolors.to_hex([r/255, g/255, b/255]) for r, g, b in centers]

#     # Sort colors by hue for better visualization
#     hsv_colors = [mcolors.rgb_to_hsv([r/255, g/255, b/255]) for r, g, b in centers]
#     sorted_indices = sorted(range(len(hsv_colors)), key=lambda i: (hsv_colors[i][0], hsv_colors[i][1], hsv_colors[i][2]))
#     sorted_hex_colors = [hex_colors[i] for i in sorted_indices]

#     # Create an HTML table for the colors
#     num_columns = 5  # Adjust as needed
#     num_rows = math.ceil(len(sorted_hex_colors) / num_columns)
#     html_table = '<table style="border-collapse: collapse;">'
#     for i in range(num_rows):
#         html_table += '<tr>'
#         for j in range(num_columns):
#             index = i * num_columns + j
#             if index < len(sorted_hex_colors):
#                 hex_color = sorted_hex_colors[index]
#                 html_table += f'<td style="background-color:{hex_color}; width:50px; height:25px; border: 1px solid #ccc;"></td>'
#                 html_table += f'<td style="padding: 5px;">{hex_color}</td>'
#         html_table += '</tr>'
#     html_table += '</table>'
    
#     # Display the HTML table with the colors for this segment
#     display(HTML(f"<h3>Movie {movie_id} - Segment {segment_id}</h3>"))
#     display(HTML(html_table))


### K_means Across Grouped Segments

In [6]:
# Group by segment_id across all movies
combined_segments = {}

for segment_id in range(num_segments):
    segment_colors = []
    for movie_id, _, centers in segment_clusters:
        if segment_id == _:
            segment_colors.append(centers)
    
    # Combine all colors from this segment across movies into a single array
    combined_segments[segment_id] = np.vstack(segment_colors)

In [7]:
# Define the number of clusters for each combined segment
num_clusters_per_combined_segment = 30  # Adjust this as needed

# Store the final cluster centers for each combined segment
final_segment_clusters = {}

for segment_id, colors in combined_segments.items():
    if len(colors) >= num_clusters_per_combined_segment:
        # Perform K-Means clustering on the combined colors for this segment
        kmeans = KMeans(n_clusters=num_clusters_per_combined_segment, n_init='auto', random_state=42)
        kmeans.fit(colors)
        
        # Store the cluster centers for this segment
        final_segment_clusters[segment_id] = kmeans.cluster_centers_
    else:
        print(f"Skipping combined segment {segment_id} due to insufficient data points.")

### Visualizing Colors Across Segments

In [8]:
# Visualize the colors for each combined segment
for segment_id, centers in final_segment_clusters.items():
    # Convert RGB values to HEX
    hex_colors = [mcolors.to_hex([r/255, g/255, b/255]) for r, g, b in centers]

    # Sort colors by hue for better visualization
    hsv_colors = [mcolors.rgb_to_hsv([r/255, g/255, b/255]) for r, g, b in centers]
    sorted_indices = sorted(range(len(hsv_colors)), key=lambda i: (hsv_colors[i][0], hsv_colors[i][1], hsv_colors[i][2]))
    sorted_hex_colors = [hex_colors[i] for i in sorted_indices]

    # Create an HTML table for the colors
    num_columns = 5  # Adjust as needed
    num_rows = math.ceil(len(sorted_hex_colors) / num_columns)
    html_table = '<table style="border-collapse: collapse;">'
    for i in range(num_rows):
        html_table += '<tr>'
        for j in range(num_columns):
            index = i * num_columns + j
            if index < len(sorted_hex_colors):
                hex_color = sorted_hex_colors[index]
                html_table += f'<td style="background-color:{hex_color}; width:50px; height:25px; border: 1px solid #ccc;"></td>'
                html_table += f'<td style="padding: 5px;">{hex_color}</td>'
        html_table += '</tr>'
    html_table += '</table>'
    
    # Display the HTML table with the colors for this segment
    display(HTML(f"<h3>Combined Segment {segment_id}</h3>"))
    display(HTML(html_table))

,#91201a,,#ae4d35,,#13110f,,#a06f48,,#73573e
,#3d352c,,#56493a,,#8d7c66,,#8d877e,,#22201d
,#b8965d,,#beb094,,#a59c86,,#bfbeab,,#67685e
,#d0d6bf,,#66bc5a,,#d9e0d8,,#6c7776,,#a3adad
,#24e8e5,,#2dc4c3,,#3b8788,,#74a7b1,,#334e55
,#6a878f,,#4f575a,,#bccbd2,,#2a7ea4,,#8b98a0


,#ad4535,,#a86347,,#4d3a27,,#8a6f53,,#6f5940
,#1b1915,,#9c8764,,#4c483d,,#b7af9d,,#afa27c
,#d2bc5d,,#b29e35,,#cec8a6,,#302f2c,,#89887c
,#6e7957,,#c1cac0,,#d7e0d9,,#959d98,,#545e58
,#6c7471,,#4dc7c4,,#9ab1b2,,#53959c,,#32838e
,#728e96,,#335461,,#697bbd,,#b76275,,#e64a5d


,#c04022,,#a95c49,,#61452e,,#a56223,,#4b3e32
,#745d44,,#362e26,,#82786a,,#1a1816,,#a59378
,#aa8f61,,#927b50,,#c4b78e,,#cbc5b2,,#a19e94
,#65675c,,#858980,,#7cac58,,#a4b1a8,,#4d524f
,#ccd8d1,,#56b2a6,,#e1e3e4,,#597073,,#3b859d
,#2f434c,,#788f99,,#2c576d,,#8ea3b3,,#a6bcce


,#b54134,,#8d6148,,#9b552c,,#2e2823,,#533b2a
,#b5a089,,#5b5044,,#967e5f,,#403b34,,#b69559
,#d0b062,,#191815,,#7b7359,,#cac3ac,,#9a9176
,#66655d,,#7ca828,,#dfe7e0,,#747976,,#d0dad3
,#b0bcb6,,#9ba39f,,#848e8c,,#7fa5a4,,#388898
,#2e444a,,#435a61,,#648693,,#85a8d4,,#bb5353


,#a74d37,,#a16455,,#b6805c,,#493726,,#69543e
,#534739,,#897c69,,#a39378,,#26231e,,#171513
,#cab690,,#7e7050,,#b0851f,,#e3be3e,,#aaa895
,#cdcebf,,#31322f,,#646863,,#515452,,#858d8a
,#d8e1de,,#76c0d0,,#64777c,,#32464b,,#688d96
,#455d63,,#95a4a8,,#b0b8bb,,#3184a3,,#bc4ec4
